In [13]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import rockphypy as rp

In [14]:
df = pd.read_csv('../Datasets/DataAVO2.csv')
df.head()

,Index,Brine_Vp,Brine_Vs,Brine_p,Shale_VP,Shale_Vs,Shale_Vs.1,Gas_Vp,Gas_Vs,Gas_p
0,1,3.28,1.68,2.19,3.27,1.65,1.68,3.04,1.74,2.05
1,2,4.06,2.03,2.40,4.69,2.61,2.03,3.70,2.06,2.26
2,3,3.85,2.24,2.24,2.77,1.52,2.24,3.08,2.34,2.14
3,4,4.06,2.34,2.30,4.06,2.18,2.34,3.62,2.58,2.30
4,5,3.21,1.79,2.22,3.05,1.69,1.79,2.91,1.85,2.01


In [15]:
def zoeppritz(vp1, vs1, rho1, vp2, vs2, rho2, theta_deg):
    theta = np.deg2rad(theta_deg)
    p = np.sin(theta) / vp1  # ray parameter

    # Snell's law: transmitted angles
    cos1 = np.cos(theta)
    sin2 = p * vp2
    cos2 = np.sqrt(1 - sin2**2)

    # Zoeppritz matrix entries (Aki & Richards style)
    a = rho2 * (1 - 2 * vs2**2 * p**2)
    b = rho1 * (1 - 2 * vs1**2 * p**2)
    c = rho2 * vs2**2
    d = rho1 * vs1**2

    E = b * cos1 + a * cos2
    F = b * sin2 - a * sin1
    G = c * cos1 + d * cos2
    H = c * sin2 - d * sin1

    # Solve linear system for Rpp, Rps, Tpp, Tps
    M = np.array([
        [E, F, -E, -F],
        [G, H, -G, -H],
        [F, -E, F, -E],
        [H, -G, H, -G],
    ])
    rhs = np.array([b * cos1 - a * cos2,
                    d * sin1 - c * sin2,
                    b * sin1 + a * sin2,
                    d * cos1 + c * cos2])

    Rpp, Rps, Tpp, Tps = np.linalg.solve(M, rhs)
    return Rpp, Rps, Tpp, Tps

In [ ]:
# Clean column names (strip whitespace/newlines)
df.columns = df.columns.str.strip()

# Convert densities to kg/m3
# Shale density is stored in 'Shale_Vs.1' (g/cc) and fluid densities are in 'Gas_p' / 'Brine_p'
df['rho_shale'] = df['Shale_Vs.1'] * 1000
df['rho_gas'] = df['Gas_p'] * 1000
df['rho_brine'] = df['Brine_p'] * 1000

theta = 30  # incident angle in degrees

rpp_shale_gas, rps_shale_gas = [], []
rpp_shale_brine, rps_shale_brine = [], []

for _, row in df.iterrows():
    # Shale-Gas interface
    Rpp, Rps = rp.AVO.zoeppritz(
        row['Shale_VP'], row['Shale_Vs'], row['rho_shale'],
        row['Gas_Vp'], row['Gas_Vs'], row['rho_gas'],
        theta
    )
    rpp_shale_gas.append(Rpp)
    rps_shale_gas.append(Rps)

    # Shale-Brine interface
    Rpp_b, Rps_b = rp.AVO.zoeppritz(
        row['Shale_VP'], row['Shale_Vs'], row['rho_shale'],
        row['Brine_Vp'], row['Brine_Vs'], row['rho_brine'],
        theta
    )
    rpp_shale_brine.append(Rpp_b)
    rps_shale_brine.append(Rps_b)

# Save to dataframe
df['Rpp_shale_gas'] = rpp_shale_gas
df['Rps_shale_gas'] = rps_shale_gas

df['Rpp_shale_brine'] = rpp_shale_brine
df['Rps_shale_brine'] = rps_shale_brine

# Compute Rp - Rs for each interface
df['Rp_minus_Rs_shale_gas'] = df['Rpp_shale_gas'] - df['Rps_shale_gas']
df['Rp_minus_Rs_shale_brine'] = df['Rpp_shale_brine'] - df['Rps_shale_brine']

# Plot Rp-Rs vs model number for both boundaries
x = df['Index']

plt.figure(figsize=(10, 5))
plt.plot(x, df['Rp_minus_Rs_shale_gas'], 'o-', label='Shale–Gas', color='navy')
plt.plot(x, df['Rp_minus_Rs_shale_brine'], 's--', label='Shale–Brine', color='teal')
plt.xlabel('Model number', fontsize=12)
plt.ylabel('Rp - Rs', fontsize=12)
plt.title(f'Rp - Rs (θ = {theta}°) for Shale–Gas and Shale–Brine', fontsize=14)
plt.grid(alpha=0.3)
plt.legend()
plt.show()

ValueError: not enough values to unpack (expected 4, got 2)